

| Secret name | Value |
|---|---|
| `DB_HOST` | e.g. `ep-xxxx.us-east-2.aws.neon.tech` (from Neon/Supabase) |
| `DB_PORT` | usually `5432` |
| `DB_NAME` | your database name |
| `DB_USER` | your database user |
| `DB_PASSWORD` | your database password |
| `JWT_SECRET` | any long random string (generate one in the next cell) |
| `SMTP_EMAIL` | your Gmail address |
| `SMTP_APP_PASSWORD` | 16-character Gmail **App Password** (not your real password) |
| `NGROK_AUTHTOKEN` | from https://dashboard.ngrok.com/get-started/your-authtoken |

**Note:** the FastAPI backend added below reuses `JWT_SECRET` — no additional secrets are needed for it.


In [1]:
!pip install -q streamlit psycopg2-binary PyJWT bcrypt python-dotenv email-validator pyngrok fastapi uvicorn python-multipart requests langdetect ftfy emoji deep-translator vaderSentiment spacy pandas matplotlib
!python -m spacy download xx_sent_ud_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 17.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 76.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 113.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('

## New: Employee Wellness NLP Analysis

After login, the app now has an **"Run NLP Analysis"** button next to the file upload. It sends the CSV/TXT to a new `/analyze` endpoint on the FastAPI backend, which detects language (Telugu/Kannada-aware), cleans and tokenizes the text, translates it to English, lemmatizes, and runs VADER sentiment + keyword-based emotion detection. Results (including sentiment/emotion bar charts) render inline in Streamlit.

**No new secrets needed** — this reuses `JWT_SECRET` like the upload feature already did.

**Heads-up:** installing spaCy + downloading the `xx_sent_ud_sm` model adds a few minutes to Section 2's install cell the first time you run it in a fresh Colab runtime.


In [2]:
from google.colab import userdata

required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []
for key in required_secrets:
    try:
        values[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing Colab secrets: {missing}. "
        f"Add them via the key icon in the left sidebar, then re-run this cell."
    )

env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10
'''

with open(".env", "w") as f:
    f.write(env_content)

print("Wrote .env with", len(values), "secrets loaded.")

Wrote .env with 9 secrets loaded.


In [3]:
%%writefile db.py
import os, psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv
load_dotenv()

CFG = dict(host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT", "5432"),
           dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
           password=os.getenv("DB_PASSWORD"), sslmode="require")

@contextmanager
def cursor(commit=False):
    conn = psycopg2.connect(**CFG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        if commit: conn.commit()
    finally:
        cur.close(); conn.close()

def init_db():
    with cursor(commit=True) as cur:
        cur.execute("""CREATE TABLE IF NOT EXISTS users (
            id SERIAL PRIMARY KEY, username VARCHAR(50) UNIQUE, email VARCHAR(255) UNIQUE,
            password_hash VARCHAR(255), is_verified BOOLEAN DEFAULT FALSE)""")
        cur.execute("""CREATE TABLE IF NOT EXISTS otp_codes (
            id SERIAL PRIMARY KEY, email VARCHAR(255), code VARCHAR(6),
            purpose VARCHAR(20), expires_at TIMESTAMP, used BOOLEAN DEFAULT FALSE)""")

Writing db.py


In [4]:
%%writefile auth.py
import os, jwt, bcrypt, random, string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from db import cursor
load_dotenv()

SECRET = os.getenv("JWT_SECRET")

def hash_pw(pw): return bcrypt.hashpw(pw.encode(), bcrypt.gensalt()).decode()
def check_pw(pw, h): return bcrypt.checkpw(pw.encode(), h.encode())

def make_token(user):
    payload = {"id": user["id"], "username": user["username"], "email": user["email"],
               "exp": datetime.now(timezone.utc) + timedelta(hours=1)}
    return jwt.encode(payload, SECRET, algorithm="HS256")

def read_token(token):
    try: return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError: return None

def get_user(email):
    with cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email=%s", (email,))
        return cur.fetchone()

def username_taken(username):
    with cursor() as cur:
        cur.execute("SELECT 1 FROM users WHERE username=%s", (username,))
        return cur.fetchone() is not None

def create_user(username, email, pw):
    with cursor(commit=True) as cur:
        cur.execute("INSERT INTO users (username,email,password_hash) VALUES (%s,%s,%s)",
                    (username, email, hash_pw(pw)))

def verify_user(email):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified=TRUE WHERE email=%s", (email,))

def set_password(email, pw):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET password_hash=%s WHERE email=%s", (hash_pw(pw), email))

def new_otp():
    return "".join(random.choices(string.digits, k=6))

def save_otp(email, code, purpose):
    exp = datetime.now(timezone.utc) + timedelta(minutes=10)
    with cursor(commit=True) as cur:
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE email=%s AND purpose=%s", (email, purpose))
        cur.execute("INSERT INTO otp_codes (email,code,purpose,expires_at) VALUES (%s,%s,%s,%s)",
                    (email, code, purpose, exp))

def check_otp(email, code, purpose):
    with cursor(commit=True) as cur:
        cur.execute("""SELECT * FROM otp_codes WHERE email=%s AND purpose=%s AND used=FALSE
                       ORDER BY id DESC LIMIT 1""", (email, purpose))
        row = cur.fetchone()
        if not row or row["code"] != code:
            return False
        now = datetime.now(row["expires_at"].tzinfo) if row["expires_at"].tzinfo else datetime.now()
        if now > row["expires_at"]:
            return False
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE id=%s", (row["id"],))
        return True

Writing auth.py


In [5]:
%%writefile email_utils.py
import os, smtplib
from email.mime.text import MIMEText
from dotenv import load_dotenv
load_dotenv()

HOST, PORT = "smtp.gmail.com", 587
EMAIL = os.getenv("SMTP_EMAIL")
APP_PW = os.getenv("SMTP_APP_PASSWORD")

def send_otp(to_email, code, purpose):
    subject = "Your Verification Code" if purpose == "signup" else "Your Password Reset Code"
    msg = MIMEText(f"Your code is: {code}\nExpires in 10 minutes.")
    msg["From"], msg["To"], msg["Subject"] = EMAIL, to_email, subject
    try:
        with smtplib.SMTP(HOST, PORT, timeout=15) as s:
            s.starttls()
            s.login(EMAIL, APP_PW)
            s.sendmail(EMAIL, to_email, msg.as_string())
        return True, "sent"
    except Exception as e:
        return False, str(e)

Writing email_utils.py


In [6]:
%%writefile app.py
import os, re, requests, streamlit as st
from db import init_db
from auth import (make_token, read_token, get_user, username_taken, create_user,
                   verify_user, set_password, check_pw, new_otp, save_otp, check_otp)
from email_utils import send_otp

st.set_page_config(page_title="Mood Mentor")

# URL of the FastAPI backend. Set BACKEND_URL as an env var / secret in deployment;
# defaults to localhost for local dev (e.g. running `uvicorn backend:app` alongside Streamlit).
BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

@st.cache_resource
def setup(): init_db()
setup()

if "page" not in st.session_state:
    st.session_state.page = "home"

if "token" not in st.session_state:
    st.session_state.token = None

if "email" not in st.session_state:
    st.session_state.email = None

if "analysis_result" not in st.session_state:
    st.session_state.analysis_result = None

def goto(p): st.session_state.page = p; st.rerun()

def valid_pw(pw):
    return len(pw) >= 8 and re.search(r"[A-Za-z]", pw) and re.search(r"[0-9]", pw)

# Main page rendering logic using if/elif
# Redirect to dashboard if logged in but on a non-dashboard/report page
if st.session_state.token and st.session_state.page not in ["dashboard", "report"]:
    st.session_state.page = "dashboard"
    st.rerun() # Rerun to render the correct page immediately

# ---- Dashboard (Logged-in view: file upload + NLP analysis page) ----
if st.session_state.page == "dashboard": # Only display dashboard if on 'dashboard' page
    # Check if user is actually logged in, if not, redirect to home
    user = read_token(st.session_state.token)
    if not user:
        st.session_state.token = None # Clear invalid token
        goto("home") # Redirect to home if token is invalid or missing
    else:
        st.title("📁 Employee Wellness Analysis")
        st.caption(f"Logged in as {user['username']} ({user['email']})")
        if st.button("Log out"):
            st.session_state.token = None; goto("home") # Go to home page after logout

        headers = {"Authorization": f"Bearer {st.session_state.token}"} # Define headers here once

        # Create two columns for the layout
        col_upload, col_chatbot = st.columns(2)

        with col_upload:
            st.subheader("📊 File Upload & NLP Analysis") # Added subheader
            uploaded = st.file_uploader("Choose a CSV or TXT file", type=["csv", "txt"])

            if uploaded is not None:
                is_csv = uploaded.name.lower().endswith(".csv")
                column_name = None
                if is_csv:
                    column_name = st.text_input(
                        "Feedback column name (leave blank to use the last column)"
                    ).strip() or None

                # Inner columns for the two buttons within the upload column
                upload_btn_col1, upload_btn_col2 = st.columns(2)

                with upload_btn_col1:
                    if st.button("Upload & Preview"):
                        files = {"file": (uploaded.name, uploaded.getvalue())}
                        try:
                            resp = requests.post(f"{BACKEND_URL}/upload", files=files,
                                                  headers=headers, timeout=15)
                        except requests.exceptions.RequestException as e:
                            st.error(f"Could not reach backend: {e}")
                            resp = None

                        if resp is not None:
                            if resp.status_code == 200:
                                data = resp.json()
                                st.success(f"Uploaded '{data['filename']}' — {data['row_count']} data row(s).")
                                if data["type"] == "csv" and data["columns"]:
                                    st.write("**Columns:**", ", ".join(data["columns"]))
                                    if data["preview_rows"]:
                                        st.dataframe(
                                            [dict(zip(data["columns"], row)) for row in data["preview_rows"]]
                                        )
                                elif data["preview_lines"]:
                                    st.text("\n".join(data["preview_lines"]))
                            else:
                                try:
                                    st.error(resp.json().get("detail", "Upload failed."))
                                except ValueError:
                                    st.error(f"Upload failed (status {resp.status_code}).")

                with upload_btn_col2:
                    if st.button("Run NLP Analysis"):
                        files = {"file": (uploaded.name, uploaded.getvalue())}
                        form = {"column": column_name} if column_name else {}
                        with st.spinner("Running multilingual NLP pipeline… this can take a moment."):
                            try:
                                resp = requests.post(f"{BACKEND_URL}/analyze", files=files, data=form,
                                                      headers=headers, timeout=120)
                            except requests.exceptions.RequestException as e:
                                st.error(f"Could not reach backend: {e}")
                                resp = None

                        if resp is not None:
                            if resp.status_code != 200:
                                try:
                                    st.error(resp.json().get("detail", "Analysis failed."))
                                except ValueError:
                                    st.error(f"Analysis failed (status {resp.status_code}).")
                            else:
                                r = resp.json()
                                st.session_state.analysis_result = r
                                st.session_state.page = "report"
                                st.rerun()

        with col_chatbot:
            st.subheader("💬 Employee Wellness Chatbot") # Moved and added subheader
            user_message = st.text_area(
                "Type your message here...", key="chatbot_input_area" # Added key
            )

            if st.button("Send", key="chatbot_send_button"): # Added key
                msg = user_message.lower()

                if any(word in msg for word in ["stress", "stressed", "pressure", "tired"]):
                    st.warning("💙 You seem stressed. Take a 5-minute break, drink some water, and try deep breathing. You've got this! 🌿")

                elif any(word in msg for word in ["happy", "great", "good", "excellent"]):
                    st.success("😊 Great to hear you're feeling good! Keep up the positive attitude.")

                else:
                    st.info("🙂 Thanks for sharing. Remember to maintain a healthy work-life balance.")

elif st.session_state.page == "report": # Report page
    r = st.session_state.analysis_result
    if not r: # If no analysis result, redirect to dashboard
        goto("dashboard")
    else:
        st.title("Employee Wellness Analysis Report")

        st.write(f"**Filename:**", {r["filename"]})
        st.write(f"**File type:**", {r["file_type"]})

        if r.get("used_column"):
            st.write("**Column used:**", r["used_column"])

        st.write("**Detected language:**",
                 r["detected_language"],
                 f"({r['language_code']})")

        with st.expander("Cleaned Text"):
            st.write(r["cleaned_text"])

        with st.expander("Sentences"):
            for i, s in enumerate(r["sentences"], 1):
                st.write(f"{i}. {s}")

        with st.expander("Tokens"):
            st.write("Original:", r["original_tokens"])
            st.write("Filtered:", r["filtered_tokens"])

        with st.expander("Translation & Lemmatization"):
            st.write("English:", r["translated_text"])
            st.write("Lemmatized:", r["lemmatized_text"])

        st.write("**Emojis:**",
                 ", ".join(r["emoji_list"]) if r["emoji_list"] else "None")

        scores = r["sentiment_scores"]

        st.subheader(f"Sentiment: {r['final_sentiment']}")
        st.bar_chart({
            "Positive": scores["pos"],
            "Negative": scores["neg"],
            "Neutral": scores["neu"]
        })

        st.caption(f"Compound Score: {scores['compound']:.3f}")

        st.subheader(f"Emotion: {r['final_emotion']}")
        st.bar_chart(r["emotion_scores"])

        st.write({
            "Characters": r["original_char_count"],
            "Sentences": len(r["sentences"]),
            "Original Tokens": len(r["original_tokens"]),
            "Filtered Tokens": len(r["filtered_tokens"]),
            "Emojis": len(r["emoji_list"])
        })

        if st.button("⬅ Back"):
            st.session_state.page = "dashboard" # Go back to dashboard (upload page)
            st.rerun()

# --- Public facing pages (home, signup, etc.) ---#
elif st.session_state.page == "home":
    st.title("Mood Mentor") # Only show title on home page
    with st.form("login"):
        email = st.text_input("Email")
        pw = st.text_input("Password", type="password")
        go = st.form_submit_button("Log in")
    if go:
        user = get_user(email.strip().lower())
        if not user or not check_pw(pw, user["password_hash"]):
            st.error("Invalid email or password.")
        elif not user["is_verified"]:
            st.warning("Verify your email first.")
            st.session_state.email = user["email"]; goto("verify")
        else:
           st.session_state.token = make_token(user)
           st.session_state.page = "dashboard" # Redirect to dashboard after login
           st.rerun()

    c1, c2 = st.columns(2)
    if c1.button("Sign up"): goto("signup")
    if c2.button("Forgot password?"): goto("forgot")

elif st.session_state.page == "signup":
    st.title("Mood Mentor - Sign Up") # Add title for clarity
    with st.form("signup"):
        username = st.text_input("Username")
        email = st.text_input("Email")
        pw = st.text_input("Password", type="password")
        go = st.form_submit_button("Create account")
    if go:
        email = email.strip().lower()
        if len(username) < 3:
            st.error("Username too short.")
        elif not valid_pw(pw):
            st.error("Password needs 8+ chars, letters and numbers.")
        elif username_taken(username) or get_user(email):
            st.error("Username or email already in use.")
        else:
            create_user(username, email, pw)
            code = new_otp(); save_otp(email, code, "signup")
            ok, msg = send_otp(email, code, "signup")
            if ok:
                st.session_state.email = email
                st.success("Check your email for the code.")
                goto("verify")
            else:
                st.error(f"Email failed: {msg}")
    if st.button("← Back"): goto("home") # Go back to home, not login

elif st.session_state.page == "verify":
    st.title("Mood Mentor - Verify Email") # Add title
    email = st.session_state.email
    st.write(f"Enter the code sent to **{email}**")
    with st.form("verify"):
        code = st.text_input("Code", max_chars=6)
        go = st.form_submit_button("Verify")
    if go:
        if check_otp(email, code.strip(), "signup"):
            verify_user(email)
            st.success("Verified! Please log in.")
            goto("home") # Go to home for login
        else:
            st.error("Invalid or expired code.")
    if st.button("← Back"): goto("signup") # Go back to signup

elif st.session_state.page == "forgot":
    st.title("Mood Mentor - Forgot Password") # Add title
    with st.form("forgot"):
        email = st.text_input("Your account email")
        go = st.form_submit_button("Send reset code")
    if go:
        email = email.strip().lower()
        if get_user(email):
            code = new_otp(); save_otp(email, code, "password_reset")
            send_otp(email, code, "password_reset")
        st.session_state.email = email
        st.info("If that email exists, a code was sent.")
        goto("reset")
    if st.button("← Back"): goto("home") # Go back to home

elif st.session_state.page == "reset":
    st.title("Mood Mentor - Reset Password") # Add title
    email = st.session_state.email
    with st.form("reset"):
        code = st.text_input("Reset code", max_chars=6)
        pw = st.text_input("New password", type="password")
        go = st.form_submit_button("Reset")
    if go:
        if not valid_pw(pw):
            st.error("Password needs 8+ chars, letters and numbers.")
        elif not check_otp(email, code.strip(), "password_reset"):
            st.error("Invalid or expired code.")
        else:
            set_password(email, pw)
            st.success("Password reset. Please log in.")
            goto("home") # Go to home for login
    if st.button("← Back"): goto("forgot") # Go back to forgot password

Writing app.py


## FastAPI backend (JWT-protected file upload)

This backend exposes a `/upload` endpoint that only accepts `.csv` or `.txt` files, verifies the same JWT issued at Streamlit login, and returns a preview (columns + first rows for CSV, first lines for TXT).


## Multilingual NLP pipeline module

Language detection, text cleaning, Telugu/Kannada stopword filtering, translation to English, lemmatization, VADER sentiment, and keyword-based emotion detection — imported by `backend.py`'s `/analyze` endpoint.


In [7]:
%%writefile nlp_pipeline.py
"""
nlp_pipeline.py
Multilingual (Telugu/Kannada-aware) NLP pipeline for employee feedback:
normalize -> detect language -> clean -> tokenize -> stopword-filter ->
translate to English -> lemmatize -> sentiment (VADER) -> emotion keywords.

Heavy libs (spacy model, translator, vader) load once at import time via
lru_cache-style module-level globals, so repeated /analyze calls reuse them.
"""

import re
import ftfy
import emoji
import spacy
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DetectorFactory.seed = 0

_nlp = None
_vader = None

LANGUAGE_NAMES = {
    "te": "Telugu", "kn": "Kannada", "en": "English", "ta": "Tamil",
    "hi": "Hindi", "ml": "Malayalam", "mr": "Marathi", "bn": "Bengali", "gu": "Gujarati",
}

TELUGU_STOPWORDS = {
    "నేను", "నాకు", "నా", "మేము", "మా", "మీరు", "మీ", "అతను", "ఆమె", "వారు",
    "ఇది", "అది", "ఇవి", "అవి", "లో", "కి", "కు", "తో", "పై", "నుండి",
    "మరియు", "కాని", "అయితే", "కూడా", "ఒక", "ఈ", "ఆ", "ఏ", "చాలా",
    "ఉంది", "ఉన్నాను", "ఉన్నారు", "అని", "కోసం", "వద్ద", "వలన",
}

KANNADA_STOPWORDS = {
    "ನಾನು", "ನನಗೆ", "ನನ್ನ", "ನಾವು", "ನಮ್ಮ", "ನೀವು", "ನಿಮ್ಮ", "ಅವನು", "ಅವಳು", "ಅವರು",
    "ಇದು", "ಅದು", "ಇವು", "ಅವು", "ನಲ್ಲಿ", "ಗೆ", "ಜೊತೆ", "ಮೇಲೆ", "ಇಂದ",
    "ಮತ್ತು", "ಆದರೆ", "ಆದರೂ", "ಕೂಡ", "ಒಂದು", "ಈ", "ಆ", "ಯಾವ", "ತುಂಬಾ",
    "ಇದೆ", "ಇದ್ದೇನೆ", "ಇದ್ದಾರೆ", "ಎಂದು", "ಗಾಗಿ", "ಬಳಿ", "ಮೂಲಕ",
}

STOPWORD_DICT = {"te": TELUGU_STOPWORDS, "kn": KANNADA_STOPWORDS}

EMOTION_KEYWORDS = {
    "Happy 😊": ["happy", "joy", "enjoy", "love", "excellent", "good"],
    "Sad 😢": ["sad", "unhappy", "lonely", "cry", "depressed"],
    "Stress 😫": ["stress", "pressure", "overwhelmed", "workload"],
    "Angry 😡": ["angry", "anger", "annoyed", "irritated"],
    "Fear 😨": ["fear", "afraid", "scared", "worried"],
}


def _get_nlp():
    """Lazy-load the multilingual spaCy model once per process."""
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("xx_sent_ud_sm")
    return _nlp


def _get_vader():
    global _vader
    if _vader is None:
        _vader = SentimentIntensityAnalyzer()
    return _vader


def process_employee_feedback(text: str) -> dict:
    """Runs the full pipeline on a single blob of text and returns a results dict."""
    nlp = _get_nlp()
    vader = _get_vader()

    normalized_text = ftfy.fix_text(text)

    try:
        language = detect(normalized_text)
    except Exception:
        language = "unknown"
    detected_language = LANGUAGE_NAMES.get(language, "Other / Unknown")

    emoji_list = [ch for ch in normalized_text if ch in emoji.EMOJI_DATA]

    cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", normalized_text)
    cleaned_text = re.sub(r"\S+@\S+", " ", cleaned_text)
    cleaned_text = re.sub(r"@\w+|#\w+", " ", cleaned_text)
    cleaned_text = emoji.replace_emoji(cleaned_text, replace="")
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    doc = nlp(cleaned_text)
    sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
    original_tokens = [t.text for t in doc if not t.is_space]
    clean_tokens = [t.text for t in doc if not t.is_punct and not t.is_space and not t.like_num]

    selected_stopwords = STOPWORD_DICT.get(language, set())
    filtered_tokens = [t for t in clean_tokens if t.lower() not in selected_stopwords]
    final_preprocessed_text = " ".join(filtered_tokens)

    try:
        translated_text = GoogleTranslator(source="auto", target="en").translate(final_preprocessed_text)
    except Exception as error:
        translated_text = f"Translation failed: {error}"

    english_doc = nlp(translated_text)
    lemmas = [t.lemma_ if t.lemma_ else t.text for t in english_doc if not t.is_space]
    lemmatized_text = " ".join(lemmas)

    sentiment_scores = vader.polarity_scores(translated_text)
    compound_score = sentiment_scores["compound"]
    if compound_score >= 0.05:
        final_sentiment = "Positive 😊"
    elif compound_score <= -0.05:
        final_sentiment = "Negative 😔"
    else:
        final_sentiment = "Neutral 😐"

    lower_translation = translated_text.lower()
    emotion_scores = {
        name: sum(1 for kw in keywords if kw in lower_translation)
        for name, keywords in EMOTION_KEYWORDS.items()
    }
    highest = max(emotion_scores.values())
    final_emotion = "Neutral 😐" if highest == 0 else max(emotion_scores, key=emotion_scores.get)

    return {
        "language_code": language,
        "detected_language": detected_language,
        "normalized_text": normalized_text,
        "cleaned_text": cleaned_text,
        "sentences": sentences,
        "original_tokens": original_tokens,
        "filtered_tokens": filtered_tokens,
        "emoji_list": emoji_list,
        "final_preprocessed_text": final_preprocessed_text,
        "translated_text": translated_text,
        "lemmatized_text": lemmatized_text,
        "sentiment_scores": sentiment_scores,
        "final_sentiment": final_sentiment,
        "emotion_scores": emotion_scores,
        "final_emotion": final_emotion,
    }


Writing nlp_pipeline.py


In [8]:
%%writefile backend.py
import os, io, jwt, csv
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
from nlp_pipeline import process_employee_feedback
load_dotenv()

SECRET = os.getenv("JWT_SECRET")
app = FastAPI(title="Upload API")

app.add_middleware(CORSMiddleware, allow_origins=["*"],
                    allow_methods=["*"], allow_headers=["*"])

def get_user(authorization: str = Header(None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing token")
    token = authorization.split(" ", 1)[1]
    try:
        return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(401, "Invalid or expired token")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/upload")
async def upload(file: UploadFile = File(...), authorization: str = Header(None)):
    user = get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024  # 5 MB cap
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    lines = text.splitlines()
    row_count = len(lines)
    preview_lines = lines[:20]

    columns = None
    preview_rows = None
    if ext == "csv":
        reader = csv.reader(io.StringIO(text))
        rows = list(reader)
        if rows:
            columns = rows[0]
            preview_rows = rows[1:21]
            row_count = max(len(rows) - 1, 0)  # exclude header from data row count

    return {
        "filename": name,
        "type": ext,
        "uploaded_by": user["username"],
        "row_count": row_count,
        "columns": columns,
        "preview_rows": preview_rows,
        "preview_lines": None if ext == "csv" else preview_lines,
    }


def _extract_text_blob(raw: bytes, ext: str, column: str | None) -> tuple[str, str | None]:
    """
    Returns (text_blob, used_column). For TXT, used_column is None.
    For CSV, joins all non-empty values of the chosen column (or the last
    column if none/invalid was specified) into one whitespace-joined blob —
    matches the notebook's "whole file as one blob" behavior.
    """
    text = raw.decode("utf-8")

    if ext == "txt":
        return text.strip(), None

    reader = csv.reader(io.StringIO(text))
    rows = list(reader)
    if not rows:
        raise HTTPException(400, "CSV file has no rows.")

    header = rows[0]
    data_rows = rows[1:]
    if not data_rows:
        raise HTTPException(400, "CSV file has a header but no data rows.")

    col_index = None
    if column and column in header:
        col_index = header.index(column)
    else:
        col_index = len(header) - 1  # default to last column

    values = [row[col_index] for row in data_rows if len(row) > col_index and row[col_index].strip()]
    blob = " ".join(values).strip()
    if not blob:
        raise HTTPException(400, f"Column '{header[col_index]}' has no readable text.")
    return blob, header[col_index]


@app.post("/analyze")
async def analyze(file: UploadFile = File(...), column: str = Form(None),
                   authorization: str = Header(None)):
    """
    Runs the multilingual NLP pipeline (language detection, cleaning,
    stopword filtering, translation, lemmatization, VADER sentiment,
    keyword-based emotion) on an uploaded .csv or .txt file.
    """
    get_user(authorization)  # just verifies the token; raises 401 if invalid

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text_blob, used_column = _extract_text_blob(raw, ext, column)
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    results = process_employee_feedback(text_blob)
    results["filename"] = name
    results["file_type"] = ext.upper()
    results["used_column"] = used_column
    results["original_char_count"] = len(text_blob)
    return results


Writing backend.py


In [9]:
from db import init_db
init_db()
print("✅ Connected to PostgreSQL and ensured tables exist.")

✅ Connected to PostgreSQL and ensured tables exist.


In [12]:
from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = values["NGROK_AUTHTOKEN"]

# Kill any previous tunnels/streamlit/uvicorn instances from earlier runs in this session
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
time.sleep(1)

# Launch FastAPI (backend.py) in the background on port 8000 (internal only, not tunneled)
get_ipython().system_raw(
    'uvicorn backend:app --host 0.0.0.0 --port 8000 &'
)
time.sleep(5)  # NLP libs (spaCy model etc.) take a little longer to import

# Launch Streamlit in the background, quietly, on port 8501
get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 --server.headless true '
    '--server.enableCORS false --server.enableXsrfProtection false &'
)
time.sleep(4)  # give both servers a moment to boot

public_url = ngrok.connect(8501, "http")
print(f"🚀 Your app is live at: {public_url}")
print("FastAPI backend is running internally on port 8000 (Streamlit talks to it via localhost).")
print("Open the URL above in your browser. Leave this Colab cell/runtime running to keep it up.")

🚀 Your app is live at: NgrokTunnel: "https://kung-coastland-coeditor.ngrok-free.dev" -> "http://localhost:8501"
FastAPI backend is running internally on port 8000 (Streamlit talks to it via localhost).
Open the URL above in your browser. Leave this Colab cell/runtime running to keep it up.


In [11]:
from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
print("Stopped Streamlit, FastAPI, and closed ngrok tunnel.")

Stopped Streamlit, FastAPI, and closed ngrok tunnel.
